# Benchmark: TabPFN vs. State-of-the-Art Algorithms

Comparison of **TabPFN** against standard ML baselines on the heart failure prediction task (3-class: early / late / healthy) using identical data pipeline (`load_final_data` → `preprocess_data` → `balance_data`).

### Algorithms
| Model | Type | Why |
|-------|------|-----|
| DummyClassifier | Baseline | Lower bound — shows what random guessing achieves |
| LogisticRegression | Linear | Classic baseline, interpretable, fast |
| RandomForest | Ensemble (Bagging) | Strong default, handles categoricals via encoding |
| XGBoost | Ensemble (Boosting) | State-of-the-art for tabular data |
| TabPFN | Foundation Model | Our main model — zero-shot Bayesian inference |

### Evaluation
- **Metrics**: Accuracy, F1 Macro, ROC-AUC (OvR), per-class F1
- **Robustness**: Each model trained on 20 balanced subsets (same seeds as TabPFN_v4)
- **Fair comparison**: Same train/val/test split, same features, same balancing

In [1]:
# ── Imports ──
import sys
sys.path.insert(0, '..')

import time
import json
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from tqdm.auto import tqdm

import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff

from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, recall_score, precision_score,
    classification_report, confusion_matrix, roc_curve, auc
)
from sklearn.preprocessing import label_binarize

from fs_thesis.data_loader import load_final_data
from fs_thesis.preprocessing import preprocess_data, balance_data, get_X_y

warnings.filterwarnings('ignore')

In [2]:
# ── Run-Ordner ──
_run_timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
RUN_DIR = Path(f"/Users/andrey/Repositories/fs-thesis/models/runs/benchmark_{_run_timestamp}")
PLOTS_DIR = RUN_DIR / "plots"
RESULTS_DIR = RUN_DIR / "results"
for d in [PLOTS_DIR, RESULTS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

_plot_counter = 0

def show_and_save(fig, name: str = None, width=1600, height=600):
    """fig.show() + PNG speichern."""
    global _plot_counter
    _plot_counter += 1
    filename = name or f"plot_{_plot_counter:02d}"
    path = PLOTS_DIR / f"{filename}.png"
    fig.write_image(str(path), scale=2, width=width, height=height)
    print(f"💾 {path}")
    fig.show()

print(f"📁 Run-Ordner: {RUN_DIR}")

📁 Run-Ordner: /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260306_083758


# 1. Data Pipeline (identical to TabPFN_v4)

In [3]:
# Lade Daten (identisch zu TabPFN_v4)
df = load_final_data()
df_train, df_val, df_test = preprocess_data(df)
X_val, y_val = get_X_y(df_val)
X_test, y_test = get_X_y(df_test)

# ── Subsampled Val-Set für TabPFN (zu langsam bei 35k) ──
VAL_SUBSAMPLE = 3000
rng = np.random.RandomState(42)
val_idx = rng.choice(len(y_val), size=VAL_SUBSAMPLE, replace=False)
X_val_small = X_val.iloc[val_idx].reset_index(drop=True)
y_val_small = y_val[val_idx]

print(f"Val: {len(y_val)} | Val (TabPFN subsample): {len(y_val_small)} | Test: {len(y_test)}")
print(f"Class distribution (val): {np.bincount(y_val)}")
print(f"Class distribution (val_small): {np.bincount(y_val_small)}")

Shapes -> Train: (143008, 17), Val: (35753, 17), Test: (44691, 17)
Val: 35753 | Val (TabPFN subsample): 3000 | Test: 44691
Class distribution (val): [ 1719  1304 32730]
Class distribution (val_small): [ 158  112 2730]


# 2. Define Models

All models use the **same interface**: `fit(X_train, y_train)` → `predict(X_val)` / `predict_proba(X_val)`.

For tree-based and linear models, categorical features are **one-hot encoded** (TabPFN handles them natively). The encoding is applied inside the benchmark loop.

In [4]:
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
# from tabpfn import TabPFNClassifier  # TabPFN auskommentiert, da zu langsam

# ── Feature-Typen definieren ──
FEATURE_COLS = ["gender", "anchor_age", "insurance", "language", "marital_status", "race", "admission_type", "bmi"]
CAT_COLS = ["gender", "insurance", "language", "marital_status", "race", "admission_type"]
NUM_COLS = ["anchor_age", "bmi"]

# ── Preprocessing Pipeline für sklearn-Modelle ──
# Numerical: impute missing values (median), then scale
# Categorical: impute missing values (most frequent), then one-hot encode
preprocessor = ColumnTransformer(
    transformers=[
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), NUM_COLS),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), CAT_COLS),
    ],
    remainder='drop'
)

# ── Modell-Definitionen ──
MODELS = {
    "DummyClassifier": Pipeline([
        ('prep', preprocessor),
        ('clf', DummyClassifier(strategy='stratified', random_state=42))
    ]),
    "LogisticRegression": Pipeline([
        ('prep', preprocessor),
        ('clf', LogisticRegression(max_iter=1000, multi_class='multinomial', random_state=42))
    ]),
    "RandomForest": Pipeline([
        ('prep', preprocessor),
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42, n_jobs=-1))
    ]),
    "XGBoost": Pipeline([
        ('prep', preprocessor),
        ('clf', XGBClassifier(
            n_estimators=200, max_depth=6, learning_rate=0.1,
            objective='multi:softprob', num_class=3,
            random_state=42, n_jobs=-1, verbosity=0,
            eval_metric='mlogloss'
        ))
    ]),
    # "TabPFN": None,  # TabPFN auskommentiert
}

print(f"Models: {list(MODELS.keys())}")
print(f"Features: {len(FEATURE_COLS)} ({len(NUM_COLS)} num, {len(CAT_COLS)} cat)")

Models: ['DummyClassifier', 'LogisticRegression', 'RandomForest', 'XGBoost']
Features: 8 (2 num, 6 cat)


In [5]:
print(len(y_val))

35753


# 3. Robustness Benchmark Loop

Each model is trained **20 times** with different balanced training subsets (seeds 42–61), identical to the TabPFN_v4 robustness loop. This measures performance **and** stability.

In [6]:
# ── Benchmark Configuration ──
N_LOOPS = 20
N_SAMPLES = 300  # same as TabPFN_v4

# Config speichern
config = {"n_loops": N_LOOPS, "n_samples": N_SAMPLES, "models": list(MODELS.keys()), "run_dir": str(RUN_DIR)}
json.dump(config, open(RUN_DIR / "config.json", "w"), indent=2)

all_results = []
start_total = time.time()

for model_name, pipeline in MODELS.items():
    print(f"\n{'='*60}")
    print(f"  {model_name}")
    print(f"{'='*60}")
    
    t0 = time.time()
    model_results = []
    
    for i in tqdm(range(N_LOOPS), desc=model_name):
        try:
            seed = 42 + i
            df_bal = balance_data(df_train, n_samples=N_SAMPLES, seed=seed)
            X_tr, y_tr = get_X_y(df_bal)
            # TabPFN ist auskommentiert
            from sklearn.base import clone
            clf = clone(pipeline)
            clf.fit(X_tr, y_tr)
            y_pred = clf.predict(X_val)
            y_proba = clf.predict_proba(X_val)
            y_eval = y_val
            f1_pc = f1_score(y_eval, y_pred, average=None)
            result = {
                'model': model_name,
                'run_id': i,
                'seed': seed,
                'accuracy': accuracy_score(y_eval, y_pred),
                'f1_macro': f1_score(y_eval, y_pred, average='macro'),
                'roc_auc_macro': roc_auc_score(y_eval, y_proba, multi_class='ovr', average='macro'),
                'recall_macro': recall_score(y_eval, y_pred, average='macro'),
                'precision_macro': precision_score(y_eval, y_pred, average='macro'),
                'f1_class_0_early': f1_pc[0],
                'f1_class_1_late': f1_pc[1],
                'f1_class_2_healthy': f1_pc[2],
            }
            all_results.append(result)
            model_results.append(result)
        except Exception as e:
            print(f"  ⚠️ FEHLER Run {i}: {e}")
    
    elapsed = time.time() - t0
    if model_results:
        df_m = pd.DataFrame(model_results)
        print(f"  ✅ F1={df_m['f1_macro'].mean():.4f}±{df_m['f1_macro'].std():.4f} | AUC={df_m['roc_auc_macro'].mean():.4f} | {elapsed:.1f}s")
    
    # Checkpoint
    pd.DataFrame(all_results).to_csv(RESULTS_DIR / "checkpoint.csv", index=False)

total_time = time.time() - start_total
print(f"\n{'='*60}")
print(f"  DONE in {total_time/60:.1f} min | {len(all_results)} total runs")
print(f"{'='*60}")


  DummyClassifier


DummyClassifier:   0%|          | 0/20 [00:00<?, ?it/s]

  ✅ F1=0.2106±0.0000 | AUC=0.4946 | 2.1s

  LogisticRegression


LogisticRegression:   0%|          | 0/20 [00:00<?, ?it/s]

  ✅ F1=0.3611±0.0047 | AUC=0.7619 | 2.5s

  RandomForest


RandomForest:   0%|          | 0/20 [00:00<?, ?it/s]

  ✅ F1=0.3653±0.0071 | AUC=0.7864 | 6.3s

  XGBoost


XGBoost:   0%|          | 0/20 [00:00<?, ?it/s]

  ✅ F1=0.3579±0.0063 | AUC=0.7671 | 12.3s

  DONE in 0.4 min | 80 total runs


# 4. Results & Comparison

In [7]:
# ── Summary Table ──
df_results = pd.DataFrame(all_results)
df_results.to_csv(RESULTS_DIR / "benchmark_all_runs.csv", index=False)

df_summary = df_results.groupby('model').agg(
    f1_mean=('f1_macro', 'mean'), f1_std=('f1_macro', 'std'),
    auc_mean=('roc_auc_macro', 'mean'), auc_std=('roc_auc_macro', 'std'),
    acc_mean=('accuracy', 'mean'), acc_std=('accuracy', 'std'),
    recall_mean=('recall_macro', 'mean'),
    precision_mean=('precision_macro', 'mean'),
    f1_early_mean=('f1_class_0_early', 'mean'), f1_early_std=('f1_class_0_early', 'std'),
    f1_late_mean=('f1_class_1_late', 'mean'), f1_late_std=('f1_class_1_late', 'std'),
    f1_healthy_mean=('f1_class_2_healthy', 'mean'), f1_healthy_std=('f1_class_2_healthy', 'std'),
    n_runs=('run_id', 'count'),
).reset_index().sort_values('f1_mean', ascending=False)

df_summary.to_csv(RESULTS_DIR / "benchmark_summary.csv", index=False)

# Schöne Darstellung
print("\n📊 Benchmark Summary (sorted by F1 Macro):\n")
display_cols = ['model', 'f1_mean', 'f1_std', 'auc_mean', 'auc_std', 'acc_mean', 'recall_mean', 'precision_mean']
print(df_summary[display_cols].to_string(index=False, float_format='{:.4f}'.format))


📊 Benchmark Summary (sorted by F1 Macro):

             model  f1_mean  f1_std  auc_mean  auc_std  acc_mean  recall_mean  precision_mean
      RandomForest   0.3653  0.0071    0.7864   0.0046    0.5652       0.6133          0.4015
LogisticRegression   0.3611  0.0047    0.7619   0.0046    0.5657       0.5724          0.3987
           XGBoost   0.3579  0.0063    0.7671   0.0061    0.5634       0.5870          0.3953
   DummyClassifier   0.2106  0.0000    0.4946   0.0000    0.3305       0.3274          0.3315


## 4.1 F1 Macro Comparison

In [8]:
# ── F1 Macro: Bar Chart mit Error Bars ──
model_order = df_summary.sort_values('f1_mean')['model'].tolist()

fig = px.bar(
    df_summary.sort_values('f1_mean'), 
    x='f1_mean', y='model', error_x='f1_std',
    orientation='h', 
    text=df_summary.sort_values('f1_mean').apply(
        lambda r: f"{r['f1_mean']:.1%} ± {r['f1_std']:.1%}", axis=1
    ),
    title=f'Benchmark: F1 Macro ({N_LOOPS} Runs, n_samples={N_SAMPLES})',
    labels={'f1_mean': 'F1 Macro Score', 'model': ''},
    color='model',
    color_discrete_sequence=px.colors.qualitative.Set2,
    template='plotly_white'
)
fig.update_layout(
    xaxis_tickformat='.0%', 
    showlegend=False,
    yaxis=dict(categoryorder='array', categoryarray=model_order),
    height=400
)
show_and_save(fig, "benchmark_f1_macro_comparison")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260306_083758/plots/benchmark_f1_macro_comparison.png


In [9]:
# ── F1 Macro: Violin Plot (Verteilung über Runs) ──
fig2 = px.violin(
    df_results, x='model', y='f1_macro', box=True, points='all',
    title=f'F1 Macro Distribution ({N_LOOPS} Runs per Model)',
    labels={'f1_macro': 'F1 Macro Score', 'model': ''},
    color='model',
    color_discrete_sequence=px.colors.qualitative.Set2,
    template='plotly_white',
    category_orders={'model': model_order[::-1]}
)
fig2.update_layout(yaxis_tickformat='.0%', showlegend=False, height=500)
show_and_save(fig2, "benchmark_f1_violin")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260306_083758/plots/benchmark_f1_violin.png


## 4.2 ROC-AUC Comparison

In [10]:
# ── AUC + F1 Combined: Grouped Bar ──
df_metrics_long = []
for _, row in df_summary.iterrows():
    df_metrics_long.append({'model': row['model'], 'Metric': 'F1 Macro', 'Score': row['f1_mean'], 'Std': row['f1_std']})
    df_metrics_long.append({'model': row['model'], 'Metric': 'ROC-AUC', 'Score': row['auc_mean'], 'Std': row['auc_std']})
    df_metrics_long.append({'model': row['model'], 'Metric': 'Accuracy', 'Score': row['acc_mean'], 'Std': row['acc_std']})

df_ml = pd.DataFrame(df_metrics_long)

fig3 = px.bar(
    df_ml, x='model', y='Score', color='Metric', error_y='Std',
    barmode='group',
    title=f'Benchmark: All Metrics ({N_LOOPS} Runs)',
    template='plotly_white',
    text_auto='.1%',
    color_discrete_map={'F1 Macro': '#e74c3c', 'ROC-AUC': '#3498db', 'Accuracy': '#2ecc71'},
    category_orders={'model': model_order[::-1]}
)
fig3.update_layout(yaxis_tickformat='.0%', xaxis_title=None, height=500)
show_and_save(fig3, "benchmark_all_metrics")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260306_083758/plots/benchmark_all_metrics.png


## 4.3 Per-Class F1 Comparison

In [11]:
# ── Per-Class F1: Heatmap ──
class_cols = {
    'Early (<1J)': 'f1_early_mean', 
    'Late (>1J)': 'f1_late_mean', 
    'Healthy': 'f1_healthy_mean'
}

# Matrix aufbauen
heatmap_data = []
models_sorted = df_summary.sort_values('f1_mean', ascending=False)['model'].tolist()

for model in models_sorted:
    row = df_summary[df_summary['model'] == model].iloc[0]
    heatmap_data.append([row[col] for col in class_cols.values()])

z = np.array(heatmap_data)
annot = [[f"{v:.1%}" for v in row] for row in z]

fig4 = ff.create_annotated_heatmap(
    z, 
    x=list(class_cols.keys()), 
    y=models_sorted,
    annotation_text=annot,
    colorscale='RdYlGn',
    showscale=True
)
fig4.update_layout(
    title='Per-Class F1 Score by Model',
    template='plotly_white',
    height=400
)
show_and_save(fig4, "benchmark_per_class_heatmap")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260306_083758/plots/benchmark_per_class_heatmap.png


# 5. Final Test Evaluation

Best model per algorithm (highest F1 on val) evaluated once on the **test set**.

In [12]:
# ── Bestes Modell pro Algorithmus auf Test-Set ──
test_results = []

for model_name in MODELS.keys():
    df_model = df_results[df_results['model'] == model_name]
    best_row = df_model.loc[df_model['f1_macro'].idxmax()]
    best_seed = int(best_row['seed'])
    
    # Reproduziere das beste Modell
    df_bal_best = balance_data(df_train, n_samples=N_SAMPLES, seed=best_seed)
    X_tr_best, y_tr_best = get_X_y(df_bal_best)
    
    if model_name == "TabPFN":
        clf_test = TabPFNClassifier(device='cpu', n_estimators=4)
        clf_test.fit(X_tr_best, y_tr_best)
    else:
        from sklearn.base import clone
        clf_test = clone(MODELS[model_name])
        clf_test.fit(X_tr_best, y_tr_best)
    
    y_test_pred = clf_test.predict(X_test)
    y_test_proba = clf_test.predict_proba(X_test)
    f1_pc = f1_score(y_test, y_test_pred, average=None)
    
    test_result = {
        'model': model_name,
        'best_seed': best_seed,
        'val_f1': best_row['f1_macro'],
        'test_accuracy': accuracy_score(y_test, y_test_pred),
        'test_f1_macro': f1_score(y_test, y_test_pred, average='macro'),
        'test_roc_auc': roc_auc_score(y_test, y_test_proba, multi_class='ovr', average='macro'),
        'test_f1_early': f1_pc[0],
        'test_f1_late': f1_pc[1],
        'test_f1_healthy': f1_pc[2],
    }
    test_results.append(test_result)
    
    print(f"\n📊 {model_name} (seed={best_seed}):")
    print(f"   Test: Acc={test_result['test_accuracy']:.2%} | F1={test_result['test_f1_macro']:.2%} | AUC={test_result['test_roc_auc']:.2%}")
    print(classification_report(y_test, y_test_pred, 
                                target_names=['Früh (<1J)', 'Spät (>1J)', 'Gesund']))

df_test_results = pd.DataFrame(test_results).sort_values('test_f1_macro', ascending=False)
df_test_results.to_csv(RESULTS_DIR / "test_final_results.csv", index=False)
print("\n" + df_test_results.to_string(index=False))


📊 DummyClassifier (seed=42):
   Test: Acc=33.18% | F1=21.25% | AUC=50.07%
              precision    recall  f1-score   support

  Früh (<1J)       0.05      0.33      0.08      2148
  Spät (>1J)       0.04      0.34      0.07      1630
      Gesund       0.91      0.33      0.49     40913

    accuracy                           0.33     44691
   macro avg       0.33      0.34      0.21     44691
weighted avg       0.84      0.33      0.45     44691


📊 LogisticRegression (seed=55):
   Test: Acc=58.55% | F1=37.16% | AUC=76.26%
              precision    recall  f1-score   support

  Früh (<1J)       0.16      0.59      0.26      2148
  Spät (>1J)       0.07      0.54      0.13      1630
      Gesund       0.97      0.59      0.73     40913

    accuracy                           0.59     44691
   macro avg       0.40      0.57      0.37     44691
weighted avg       0.90      0.59      0.69     44691


📊 RandomForest (seed=44):
   Test: Acc=58.38% | F1=37.39% | AUC=79.14%
             

In [13]:
# ── Test Results: Bar Chart ──
df_test_sorted = df_test_results.sort_values('test_f1_macro')

fig5 = px.bar(
    df_test_sorted,
    x='test_f1_macro', y='model', 
    orientation='h',
    text=df_test_sorted.apply(
        lambda r: f"F1={r['test_f1_macro']:.1%} | AUC={r['test_roc_auc']:.1%}", axis=1
    ),
    title='Final Test Set: F1 Macro by Model',
    labels={'test_f1_macro': 'F1 Macro Score', 'model': ''},
    color='model',
    color_discrete_sequence=px.colors.qualitative.Set2,
    template='plotly_white'
)
fig5.update_layout(xaxis_tickformat='.0%', showlegend=False, height=400)
show_and_save(fig5, "benchmark_test_f1")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260306_083758/plots/benchmark_test_f1.png


# 6. ROC Curves (Test Set, Best Models)

One-vs-Rest ROC curves for each model on the test set, all in one plot per class.

In [14]:
# ── ROC Curves: All models, Macro-averaged ──
from sklearn.preprocessing import label_binarize
from sklearn.metrics import roc_curve, auc

y_test_bin = label_binarize(y_test, classes=[0, 1, 2])
class_names = ['Früh (<1J)', 'Spät (>1J)', 'Gesund']
model_colors = {
    'DummyClassifier': '#95a5a6',
    'LogisticRegression': '#3498db',
    'RandomForest': '#2ecc71',
    'XGBoost': '#e67e22',
    'TabPFN': '#e74c3c',
}

# Re-train best models and get probabilities
best_probas = {}
for model_name in MODELS.keys():
    df_model = df_results[df_results['model'] == model_name]
    best_seed = int(df_model.loc[df_model['f1_macro'].idxmax(), 'seed'])
    
    df_bal = balance_data(df_train, n_samples=N_SAMPLES, seed=best_seed)
    X_tr, y_tr = get_X_y(df_bal)
    
    if model_name == "TabPFN":
        clf = TabPFNClassifier(device='cpu', n_estimators=4)
        clf.fit(X_tr, y_tr)
    else:
        from sklearn.base import clone
        clf = clone(MODELS[model_name])
        clf.fit(X_tr, y_tr)
    
    best_probas[model_name] = clf.predict_proba(X_test)

# Plot: One subplot per class
from plotly.subplots import make_subplots

fig6 = make_subplots(rows=1, cols=3, subplot_titles=[f'OvR: {cn}' for cn in class_names])

for i, cn in enumerate(class_names):
    for model_name, proba in best_probas.items():
        fpr, tpr, _ = roc_curve(y_test_bin[:, i], proba[:, i])
        roc_auc_val = auc(fpr, tpr)
        fig6.add_trace(
            go.Scatter(x=fpr, y=tpr, mode='lines',
                       name=f'{model_name} ({roc_auc_val:.3f})',
                       line=dict(color=model_colors.get(model_name, 'grey'), width=2),
                       showlegend=(i == 0)),  # legend only on first subplot
            row=1, col=i+1
        )
    # Diagonal
    fig6.add_trace(
        go.Scatter(x=[0,1], y=[0,1], mode='lines',
                   line=dict(color='grey', width=1, dash='dash'), showlegend=False),
        row=1, col=i+1
    )

fig6.update_layout(
    title='ROC Curves (Test Set) — One-vs-Rest per Class',
    template='plotly_white', height=500, width=1600,
    legend=dict(x=1.02, y=1)
)
for i in range(3):
    fig6.update_xaxes(title_text='FPR', row=1, col=i+1)
    fig6.update_yaxes(title_text='TPR', row=1, col=i+1)

show_and_save(fig6, "benchmark_roc_curves", width=1600, height=500)

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260306_083758/plots/benchmark_roc_curves.png


# 7. Data Quality Check: Dead but "Healthy"?

Patients classified as **target=2 (healthy/censored)** who have a recorded date of death (`dod`). These patients died but were never diagnosed with heart failure — they are **correctly** censored (not false labels), but it's important to know how many there are and whether they bias the model.

In [15]:
# ── Data Quality: Verstorbene Patienten in der "Gesund"-Klasse ──
import polars as pl

# df ist das volle Dataset (vor dem Split)
df_quality = df.to_pandas() if hasattr(df, 'to_pandas') else df

# Patienten mit target=2 (healthy/censored) UND dod vorhanden
dead_but_healthy = df.filter(
    (pl.col("target") == 2) & (pl.col("dod").is_not_null())
)

total_healthy = df.filter(pl.col("target") == 2).height
n_dead_healthy = dead_but_healthy.height

print(f"Total patients with target=2 (healthy/censored): {total_healthy:,}")
print(f"Davon mit date of death (dod): {n_dead_healthy:,} ({n_dead_healthy/total_healthy:.1%})")
print(f"Davon ohne dod (truly alive): {total_healthy - n_dead_healthy:,}")

# Verteilung der Überlebensdauer bei den "toten Gesunden"
if n_dead_healthy > 0:
    t_death_stats = dead_but_healthy.select("t_death").to_pandas()["t_death"].describe()
    print(f"\nÜberlebensdauer (t_death) der verstorbenen 'Gesunden':")
    print(t_death_stats)
    
    fig_dq = px.histogram(
        dead_but_healthy.select("t_death").to_pandas(), 
        x="t_death", nbins=50,
        title=f"Verstorbene Patienten ohne HF-Diagnose (n={n_dead_healthy:,}): Tage bis Tod",
        labels={"t_death": "Tage von Baseline bis Tod"},
        template="plotly_white"
    )
    fig_dq.add_vline(x=365, line_dash="dash", line_color="red", annotation_text="1 Jahr")
    show_and_save(fig_dq, "data_quality_dead_but_healthy")

Total patients with target=2 (healthy/censored): 204,562
Davon mit date of death (dod): 29,503 (14.4%)
Davon ohne dod (truly alive): 175,059

Überlebensdauer (t_death) der verstorbenen 'Gesunden':
count    29503.000000
mean       651.097956
std        966.801290
min      -2520.000000
25%         39.000000
50%        217.000000
75%        845.000000
max       5615.000000
Name: t_death, dtype: float64
💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260306_083758/plots/data_quality_dead_but_healthy.png


# 8. Korrelation vs. Kausalität (Feature-Analyse)

**Philipps Feedback: "Hosenträger rausnehmen"** — Features mit Korrelation aber ohne Kausalität identifizieren.

| Analyse | Was sie misst | Methode |
|---------|--------------|---------|
| **Korrelation** (univariat) | Wie stark hängt ein Feature *alleine* mit dem Target zusammen? | Cramér's V (kategorisch), Eta² (numerisch) |
| **Prädiktive Bedeutung** (multivariat) | Wie viel *eigene* Vorhersagekraft hat ein Feature, wenn alle anderen bekannt sind? | Permutation Importance (F1 Macro) |

**Interpretation:**
- Hohe Korrelation + hohe Importance → **Echter Treiber** (z.B. Alter, BMI)
- Hohe Korrelation + niedrige/negative Importance → **Hosenträger / Confounder** (z.B. Insurance korreliert mit Alter)
- Niedrige Korrelation + niedrige Importance → **Irrelevant** (kann raus)

## 8.1 Univariate Korrelation (Feature ↔ Target)

In [16]:
# ── Univariate Korrelation: Wie stark hängt jedes Feature ALLEINE mit dem Target zusammen? ──
from scipy.stats import chi2_contingency, f_oneway

def cramers_v(x, y):
    """Cramér's V: Assoziationsmaß für kategorisch × kategorisch (0 = kein Zusammenhang, 1 = perfekt)."""
    ct = pd.crosstab(x, y)
    chi2 = chi2_contingency(ct)[0]
    n = len(x)
    min_dim = min(ct.shape) - 1
    if min_dim == 0 or n == 0:
        return 0.0
    return np.sqrt(chi2 / (n * min_dim))

def eta_squared(feature_values, target_values):
    """Eta²: Effektstärke für numerisch × kategorisch (ANOVA). 0 = kein Effekt, 1 = perfekter Effekt."""
    groups = [feature_values[target_values == c].dropna() for c in sorted(target_values.unique())]
    groups = [g for g in groups if len(g) > 0]
    if len(groups) < 2:
        return 0.0
    f_stat, p_val = f_oneway(*groups)
    # Eta² = SS_between / SS_total
    grand_mean = feature_values.dropna().mean()
    ss_between = sum(len(g) * (g.mean() - grand_mean)**2 for g in groups)
    ss_total = ((feature_values.dropna() - grand_mean)**2).sum()
    if ss_total == 0:
        return 0.0
    return ss_between / ss_total

# Berechne Korrelation für jedes Feature
df_val_pd = X_val.copy()
df_val_pd['target'] = y_val

correlation_results = []
for feature in FEATURE_COLS:
    if feature in CAT_COLS:
        corr = cramers_v(df_val_pd[feature], df_val_pd['target'])
        method = "Cramér's V"
    else:
        corr = eta_squared(df_val_pd[feature], df_val_pd['target'])
        method = "Eta²"
    
    correlation_results.append({
        'Feature': feature,
        'Correlation': corr,
        'Method': method,
        'Type': 'categorical' if feature in CAT_COLS else 'numerical'
    })

df_corr = pd.DataFrame(correlation_results).sort_values('Correlation', ascending=False)

print("📊 Univariate Korrelation (Feature → Target):\n")
print(df_corr.to_string(index=False, float_format='{:.4f}'.format))

# Plot
fig_corr = px.bar(
    df_corr.sort_values('Correlation'),
    x='Correlation', y='Feature', orientation='h',
    color='Type',
    color_discrete_map={'categorical': '#3498db', 'numerical': '#e74c3c'},
    text=df_corr.sort_values('Correlation').apply(
        lambda r: f"{r['Correlation']:.3f} ({r['Method']})", axis=1
    ),
    title=f"Univariate Korrelation: Feature ↔ Target (n={len(y_val)})",
    labels={'Correlation': 'Assoziationsstärke', 'Feature': ''},
    template='plotly_white'
)
fig_corr.update_layout(height=450)
show_and_save(fig_corr, "correlation_univariate")

📊 Univariate Korrelation (Feature → Target):

       Feature  Correlation     Method        Type
admission_type       0.1327 Cramér's V categorical
     insurance       0.1313 Cramér's V categorical
          race       0.0810 Cramér's V categorical
marital_status       0.0751 Cramér's V categorical
    anchor_age       0.0523       Eta²   numerical
      language       0.0514 Cramér's V categorical
        gender       0.0409 Cramér's V categorical
           bmi       0.0080       Eta²   numerical
💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260306_083758/plots/correlation_univariate.png


## 8.2 Prädiktive Bedeutung (Permutation Importance)

Misst wie viel **eigene** Vorhersagekraft ein Feature hat, **wenn alle anderen Features bekannt sind**. Ein Feature mit hoher Korrelation aber niedriger Importance ist ein Confounder — es korreliert nur, weil es mit einem echten Treiber zusammenhängt.

In [17]:
# ── Permutation Importance: Prädiktive Bedeutung pro Feature (alle Benchmark-Modelle) ──
from sklearn.inspection import permutation_importance
from sklearn.base import clone

# Für jedes Modell: Bestes Modell reproduzieren, dann Permutation Importance berechnen
importance_results = []

for model_name in MODELS.keys():
    print(f"\n🔄 {model_name}...")
    df_model = df_results[df_results['model'] == model_name]
    best_seed = int(df_model.loc[df_model['f1_macro'].idxmax(), 'seed'])
    
    df_bal = balance_data(df_train, n_samples=N_SAMPLES, seed=best_seed)
    X_tr, y_tr = get_X_y(df_bal)
    
    if model_name == "TabPFN":
        clf = TabPFNClassifier(device='cpu', n_estimators=4)
        clf.fit(X_tr, y_tr)
        perm = permutation_importance(clf, X_val, y_val, n_repeats=10, 
                                       random_state=42, scoring='f1_macro', n_jobs=1)
    else:
        clf = clone(MODELS[model_name])
        clf.fit(X_tr, y_tr)
        perm = permutation_importance(clf, X_val, y_val, n_repeats=10, 
                                       random_state=42, scoring='f1_macro', n_jobs=-1)
    
    for j, feature in enumerate(FEATURE_COLS):
        importance_results.append({
            'Model': model_name,
            'Feature': feature,
            'Importance': perm.importances_mean[j],
            'Std': perm.importances_std[j],
        })
    print(f"  ✅ done")

df_importance = pd.DataFrame(importance_results)
df_importance.to_csv(RESULTS_DIR / "permutation_importance_all_models.csv", index=False)

# Durchschnitt über alle Modelle (ohne Dummy)
df_imp_avg = df_importance[df_importance['Model'] != 'DummyClassifier'].groupby('Feature').agg(
    Importance_mean=('Importance', 'mean'),
    Importance_std=('Importance', 'std'),
).reset_index().sort_values('Importance_mean', ascending=False)

print("\n📊 Durchschnittliche Permutation Importance (ohne Dummy):\n")
print(df_imp_avg.to_string(index=False, float_format='{:.4f}'.format))


🔄 DummyClassifier...
  ✅ done

🔄 LogisticRegression...
  ✅ done

🔄 RandomForest...
  ✅ done

🔄 XGBoost...
  ✅ done

📊 Durchschnittliche Permutation Importance (ohne Dummy):

       Feature  Importance_mean  Importance_std
    anchor_age           0.0371          0.0198
admission_type           0.0314          0.0004
           bmi           0.0150          0.0116
     insurance           0.0034          0.0024
          race           0.0031          0.0003
marital_status           0.0019          0.0031
        gender           0.0013          0.0007
      language           0.0008          0.0005


In [18]:
# ── Importance Heatmap: pro Modell × Feature ──
pivot = df_importance.pivot_table(index='Model', columns='Feature', values='Importance')
models_order = ['DummyClassifier', 'LogisticRegression', 'RandomForest', 'XGBoost', 'TabPFN']
pivot = pivot.reindex(models_order)
features_order = df_imp_avg.sort_values('Importance_mean', ascending=True)['Feature'].tolist()
pivot = pivot[features_order]

annot = [[f"{v:.3f}" for v in row] for row in pivot.values]

fig_imp_heat = ff.create_annotated_heatmap(
    pivot.values, x=features_order, y=models_order,
    annotation_text=annot, colorscale='RdBu', reversescale=False, showscale=True
)
fig_imp_heat.update_layout(
    title='Permutation Importance: Feature × Model (F1 Macro)',
    template='plotly_white', height=400
)
show_and_save(fig_imp_heat, "importance_heatmap_all_models")

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260306_083758/plots/importance_heatmap_all_models.png


## 8.3 Korrelation vs. Kausalität — Der Hosenträger-Plot

Vergleicht die univariate Korrelation (deskriptiv) mit der multivariaten Importance (prädiktiv). Features im **oberen linken Quadranten** (hohe Korrelation, niedrige Importance) sind die "Hosenträger" — sie korrelieren, weil sie Proxies für echte Treiber sind.

In [19]:
# ── Korrelation vs. Kausalität: Scatter-Plot ──
df_combined = df_corr.merge(df_imp_avg, on='Feature')

# Klassifikation der Features
def classify_feature(row):
    corr_threshold = df_combined['Correlation'].median()
    imp_threshold = 0.005  # minimal positive importance
    if row['Correlation'] >= corr_threshold and row['Importance_mean'] >= imp_threshold:
        return '✅ Echter Treiber'
    elif row['Correlation'] >= corr_threshold and row['Importance_mean'] < imp_threshold:
        return '⚠️ Hosenträger (Confounder)'
    elif row['Correlation'] < corr_threshold and row['Importance_mean'] >= imp_threshold:
        return '🔍 Versteckter Treiber'
    else:
        return '❌ Irrelevant'

df_combined['Kategorie'] = df_combined.apply(classify_feature, axis=1)

# Scatter: Korrelation (x) vs. Importance (y)
fig_scatter = px.scatter(
    df_combined, 
    x='Correlation', y='Importance_mean',
    text='Feature',
    color='Kategorie',
    color_discrete_map={
        '✅ Echter Treiber': '#2ecc71',
        '⚠️ Hosenträger (Confounder)': '#e67e22',
        '🔍 Versteckter Treiber': '#3498db',
        '❌ Irrelevant': '#95a5a6',
    },
    error_y='Importance_std',
    title='Korrelation vs. Prädiktive Bedeutung — "Hosenträger-Plot"',
    labels={
        'Correlation': 'Univariate Korrelation (Cramér\'s V / Eta²)',
        'Importance_mean': 'Permutation Importance (Ø über Modelle, F1 Macro)'
    },
    template='plotly_white'
)

# Quadranten-Linien
corr_median = df_combined['Correlation'].median()
fig_scatter.add_hline(y=0.005, line_dash="dash", line_color="grey", opacity=0.5,
                       annotation_text="Importance-Schwelle")
fig_scatter.add_vline(x=corr_median, line_dash="dash", line_color="grey", opacity=0.5,
                       annotation_text="Korrelation-Median")
fig_scatter.add_hline(y=0, line_dash="solid", line_color="black", opacity=0.3)

fig_scatter.update_traces(textposition='top center', marker=dict(size=14))
fig_scatter.update_layout(height=600, width=900)
show_and_save(fig_scatter, "correlation_vs_causality_scatter", width=900, height=600)

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260306_083758/plots/correlation_vs_causality_scatter.png


In [20]:
# ── Side-by-Side Bar: Korrelation vs. Importance ──
df_side = df_combined.sort_values('Correlation', ascending=True)

fig_side = go.Figure()

fig_side.add_trace(go.Bar(
    y=df_side['Feature'], x=df_side['Correlation'],
    name='Korrelation (univariat)', orientation='h',
    marker_color='#3498db', opacity=0.8
))

fig_side.add_trace(go.Bar(
    y=df_side['Feature'], x=df_side['Importance_mean'],
    name='Importance (multivariat)', orientation='h',
    marker_color='#e74c3c', opacity=0.8,
    error_x=dict(type='data', array=df_side['Importance_std'].values, visible=True)
))

fig_side.update_layout(
    barmode='group',
    title='Korrelation vs. Prädiktive Bedeutung (Side-by-Side)',
    xaxis_title='Stärke',
    yaxis_title=None,
    template='plotly_white',
    height=500,
    legend=dict(x=0.6, y=0.05)
)
show_and_save(fig_side, "correlation_vs_importance_bars")

# ── Ergebnis-Tabelle ──
print("\n📊 Feature-Klassifikation:\n")
display_df = df_combined[['Feature', 'Correlation', 'Method', 'Importance_mean', 'Importance_std', 'Kategorie']]
display_df = display_df.sort_values('Correlation', ascending=False)
print(display_df.to_string(index=False, float_format='{:.4f}'.format))

# Speichern
df_combined.to_csv(RESULTS_DIR / "correlation_vs_importance.csv", index=False)

# Zusammenfassung
print("\n" + "="*60)
hosentraeger = df_combined[df_combined['Kategorie'].str.contains('Hosenträger')]
treiber = df_combined[df_combined['Kategorie'].str.contains('Treiber')]
irrelevant = df_combined[df_combined['Kategorie'].str.contains('Irrelevant')]

if not treiber.empty:
    print(f"✅ Echte Treiber: {', '.join(treiber['Feature'].tolist())}")
if not hosentraeger.empty:
    print(f"⚠️  Hosenträger (Confounder): {', '.join(hosentraeger['Feature'].tolist())}")
    print(f"   → Diese Features korrelieren mit dem Target, tragen aber KEINE eigene Vorhersagekraft bei.")
    print(f"   → Sie sind Proxies für die echten Treiber (z.B. Insurance ≈ Alter).")
if not irrelevant.empty:
    print(f"❌ Irrelevant: {', '.join(irrelevant['Feature'].tolist())}")
print("="*60)

💾 /Users/andrey/Repositories/fs-thesis/models/runs/benchmark_20260306_083758/plots/correlation_vs_importance_bars.png



📊 Feature-Klassifikation:

       Feature  Correlation     Method  Importance_mean  Importance_std                   Kategorie
admission_type       0.1327 Cramér's V           0.0314          0.0004            ✅ Echter Treiber
     insurance       0.1313 Cramér's V           0.0034          0.0024 ⚠️ Hosenträger (Confounder)
          race       0.0810 Cramér's V           0.0031          0.0003 ⚠️ Hosenträger (Confounder)
marital_status       0.0751 Cramér's V           0.0019          0.0031 ⚠️ Hosenträger (Confounder)
    anchor_age       0.0523       Eta²           0.0371          0.0198       🔍 Versteckter Treiber
      language       0.0514 Cramér's V           0.0008          0.0005                ❌ Irrelevant
        gender       0.0409 Cramér's V           0.0013          0.0007                ❌ Irrelevant
           bmi       0.0080       Eta²           0.0150          0.0116       🔍 Versteckter Treiber

✅ Echte Treiber: admission_type, anchor_age, bmi
⚠️  Hosenträger (Confo

## 8.4 Feature Ablation — Bestätigung durch Experiment

Die Klassifikation aus 8.3 wird nun experimentell überprüft: Verbessert sich das Modell, wenn man die "Hosenträger" entfernt? Zusätzlich wird eine **dynamische Config** aus den automatisch erkannten Treibern erzeugt.

In [22]:
# ── Feature Ablation: TabPFN with different feature sets ──
# TabPFN ist auskommentiert, daher wird die Ablation hier übersprungen.
print("TabPFN ist deaktiviert. Feature Ablation wird übersprungen.")
# Wenn TabPFN wieder aktiviert werden soll, bitte die entsprechenden Imports und Codeblöcke wieder einkommentieren.

TabPFN ist deaktiviert. Feature Ablation wird übersprungen.


In [ ]:
# ── Ablation Visualization ──
print("TabPFN ist deaktiviert. Ablation-Visualisierung wird übersprungen.")
# Wenn TabPFN wieder aktiviert werden soll, bitte die entsprechenden Ablation-Auswertungen wieder einkommentieren.

NameError: name 'df_abl_summary' is not defined

# Summary

This notebook provides four key results for the thesis:

1. **Benchmark** (Section 3-6): TabPFN compared against DummyClassifier, LogisticRegression, RandomForest, and XGBoost — all with identical data pipeline, 20 runs each.

2. **Data Quality** (Section 7): Quantifies how many target=2 patients are actually deceased (censored correctly, but important for interpretation).

3. **Korrelation vs. Kausalität** (Section 8.1–8.3): Systematic analysis distinguishing:
   - **Univariate Korrelation** (Cramér's V / Eta²): deskriptive Assoziation
   - **Permutation Importance** (F1 Macro, alle Modelle): prädiktive Bedeutung
   - **Hosenträger-Plot**: Identifiziert Confounder (hohe Korrelation, keine eigene Vorhersagekraft)

4. **Feature Ablation** (Section 8.4): Experimentelle Bestätigung — verbessert sich TabPFN wenn Confounder entfernt werden?

All results are saved in the run folder for reproducibility.